# 04c · Compare pretrained methods on your actual industrial parts
**Use a fresh Colab T4 GPU session.** No training and no API-paid inference.
No dataset/reference photos are bundled. Set IMAGE_PATH to a Drive image/folder
or leave it blank to upload photos. First-use dependencies
and weights still require internet and several GB of local runtime disk space.

Five default methods, not five sizes of one model:
1. Larger YOLOE text-prompted segmentation (direct masks).
2. SAM 2.1 automatic grid masks (class-agnostic baseline).
3. Grounding DINO + SAM 2.1 (text detection, box-prompted masks).
4. OWLv2 + SAM 2.1 (different open-vocabulary detector, same mask decoder).
5. Florence-2 + SAM 2.1 (generative grounding, same mask decoder).
Optional sixth: SAM 3 text segmentation, requiring approved checkpoint access.

All methods see the same images and optional workspace crop. No forced six-mask
selection. No expected counts are assumed for uploaded photos.
Count agreement does NOT prove correct segmentation. Mask scores are NOT comparable between
architectures, and no accuracy winner is selected without human ground truth.
Detector-plus-SAM methods can only segment objects whose boxes were proposed.

This is a development comparison, not an independent held-out benchmark.
No method is guaranteed to segment these pieces correctly. Masks are not safe
robot grasps, and no hidden surfaces, height, defect labels or robot control are inferred.
User-supplied image rights: verify before redistribution.

Sources: [YOLOE](https://docs.ultralytics.com/models/yoloe/),
[SAM 2](https://github.com/facebookresearch/sam2),
[Grounding DINO](https://huggingface.co/IDEA-Research/grounding-dino-base),
[OWLv2](https://huggingface.co/google/owlv2-base-patch16-ensemble),
[Florence-2](https://huggingface.co/docs/transformers/v5.2.0/en/model_doc/florence2),
[SAM 3](https://huggingface.co/facebook/sam3).
Review each model's license/access terms. Coding agent has not run inference.


In [ ]:
# Change these values BEFORE running subsequent cells.
USE_DRIVE = True
ROOT = '/content/drive/MyDrive/Robotekin2026'
SEED = 42
DATASET_PATH = ''  # Existing dataset folder on Drive; empty opens upload.
MODEL_PATH = ''    # Existing exported model folder; empty opens upload.
RESUME_RUN = ''    # Existing run folder for checkpoint continuation.
IMAGE_PATH = ''  # Drive image/folder; blank uploads one or more photos
METHODS = ['yoloe_text','sam2_auto','grounding_dino_sam2','owlv2_sam2','florence2_sam2']
ENABLE_SAM3 = False  # Optional sixth; requires approved facebook/sam3 access and HF_TOKEN
PROMPTS = ['metal bracket','sheet metal part','metal mounting clip']
ROI_NORM = None  # Optional shared [left,top,right,bottom] fractions; None = whole photo
GT_MANIFEST_PATH = ''  # Optional JSON: filename -> list of full-frame binary instance-mask PNG paths
MODEL_REVISIONS = {}  # HF model ID -> pinned revision; resolved revisions are recorded
SAM2_ID = 'facebook/sam2.1-hiera-large'  # Set small for less memory, consistently across methods
SAM2_COMMIT = '2b90b9f5ceec907a1c18123530e92e794ad901a4'
YOLOE_CHECKPOINT = 'yoloe-26l-seg.pt'
IMGSZ = 960
YOLOE_THRESHOLD = 0.25
DINO_THRESHOLD = 0.25
DINO_TEXT_THRESHOLD = 0.20
OWL_THRESHOLD = 0.10
SAM3_THRESHOLD = 0.50
BOX_NMS_IOU = 0.70
MAX_DETECTIONS = 50  # Resource limit, not expected part count
POINTS_PER_SIDE = 32
POINTS_PER_BATCH = 8
CROP_N_LAYERS = 0
SAM_PRED_IOU = 0.80
SAM_STABILITY = 0.95
FLORENCE_MAX_NEW_TOKENS = 1024
MIN_MASK_AREA = 100
MAX_MASK_FRACTION = 0.85
WARMUP = True  # Extra untimed prediction on first image, per method
CPU_THREADS = 2


In [ ]:
MODEL_CACHE = '/content/roboqc_pretrained_cache'  # All model weights on local disk, NOT Drive
SAM2_CACHE = '/content/roboqc_sam2_cache'  # Local SAM cache, reused within this Colab session


SAM checkpoint recovery: SAM 2.1 weights are cached on local Colab disk,
validated with `weights_only=True`, then loaded strictly. If deserialization
fails, one new download is attempted in a fresh subfolder; previous files
are preserved. Later methods reuse the recovered SAM cache. Detector weights
also use local disk (`MODEL_CACHE`), with one fresh-cache retry on a
SafetensorError. No unsafe pickle fallback is used. Results remain on Drive;
all local model caches disappear on runtime reset and must download again.
Failed methods print their full traceback and also save it in the results.


In [ ]:
import os, random, subprocess, sys, uuid
from datetime import datetime, timezone
from pathlib import Path
print('Python:', sys.version, '| executable:', sys.executable, flush=True)
install_log = Path('/content/roboqc_pip_install.log')
install_log.parent.mkdir(parents=True, exist_ok=True)
command = [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', 'numpy>=1.26,<3', 'torch>=2.5.1,<3', 'torchvision>=0.20.1,<1', 'transformers==5.2.0', 'huggingface-hub>=1.3,<2', 'ultralytics>=8.4.0,<9', 'matplotlib>=3.9,<4', 'Pillow>=10,<13', 'opencv-python-headless>=4.10,<6', 'pandas>=2.2,<4', 'scipy>=1.13,<2', 'hydra-core>=1.3.2,<2', 'iopath>=0.1.10,<1', 'tqdm>=4.66,<5']
with install_log.open('w', encoding='utf-8') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
    exit_code = process.wait()
if exit_code:
    raise RuntimeError(f'pip failed (exit {exit_code}). Read the ERROR lines above. Full log: {install_log}')
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    ROOT = '/content/Robotekin2026'
RUN = Path(RESUME_RUN) if RESUME_RUN else Path(ROOT) / (datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6])
RUN.mkdir(parents=True, exist_ok=bool(RESUME_RUN))
import numpy as np, torch
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
GPU = torch.cuda.is_available()
DEVICE = 'cuda' if GPU else 'cpu'
print('Device:', DEVICE, '| Run:', RUN)
(RUN / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))


In [ ]:
# Same pinned official SAM 2 install as 04b; skip optional CUDA extension builds.
os.environ['SAM2_BUILD_CUDA'] = '0'
command = [sys.executable,'-m','pip','install','--no-build-isolation',
           'git+https://github.com/facebookresearch/sam2.git@' + SAM2_COMMIT]
with install_log.open('a',encoding='utf-8') as log:
    process = subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout: print(line,end='',flush=True); log.write(line)
    if process.wait(): raise RuntimeError(f'SAM install failed. See {install_log}')
(RUN / 'environment.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
torch.set_num_threads(CPU_THREADS)
if not GPU: print('WARNING: CPU detected. Five-method comparison can be very slow; choose a GPU runtime.')
CACHE = Path(MODEL_CACHE).resolve()
CACHE.mkdir(parents=True,exist_ok=True)
os.environ['HF_HOME'] = str(CACHE / 'huggingface')
if ENABLE_SAM3:
    # Store an approved token in Colab Secrets as HF_TOKEN; never print it.
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
    except Exception:
        token = os.environ.get('HF_TOKEN')
    if token:
        from huggingface_hub import login
        login(token=token,add_to_git_credential=False)
        del token
    print('SAM 3 enabled: missing access approval will be reported as a method error, not substituted.')


In [ ]:
"""Photo input helper embedded as source in public notebooks; never embeds photos."""

from pathlib import Path
import uuid


def select_image_paths(image_path, run, uploader=None):
    suffixes = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}
    if image_path:
        source = Path(image_path).resolve()
        if not source.exists():
            raise FileNotFoundError(source)
    else:
        if uploader is None:
            from google.colab import files
            uploader = files.upload
        uploaded = uploader()
        if not uploaded:
            raise ValueError('Upload one or more photos, or set IMAGE_PATH to a Drive image/folder')
        if any(Path(name).name != name or '\\' in name or ':' in name or
               Path(name).suffix.lower() not in suffixes for name in uploaded):
            raise ValueError('Upload image files only (not ZIPs); use IMAGE_PATH for nested folders')
        source = Path(run) / ('uploaded_photos_' + uuid.uuid4().hex)
        source.mkdir(parents=True, exist_ok=False)
        for name, blob in uploaded.items():
            with (source / name).open('xb') as handle:
                handle.write(blob)
    paths = [source] if source.is_file() else sorted(p for p in source.rglob('*') if p.is_file())
    paths = [p for p in paths if p.suffix.lower() in suffixes]
    if not paths:
        raise ValueError('No supported photos found; set IMAGE_PATH or upload photos')
    return paths


In [ ]:
import hashlib, json, shutil, time, gc, traceback
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
SOURCES = {'roboqc/__init__.py': '', 'roboqc/industrial/__init__.py': '', 'roboqc/industrial/vision_inference.py': '"""CPU segmentation, planar geometry and optional visible-quality classifier.\n\nThis module intentionally imports no GUI or robot library and is embedded in Colab.\n"""\n\nimport json\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\n\nCLASSES = ["bolt", "nut", "washer", "bracket"]\nQUALITY = ["good", "defect_1", "defect_2"]\n\n\ndef mask_geometry(mask, symmetric=False, clearance_px=3):\n    mask = np.asarray(mask, dtype=np.uint8)\n    ys, xs = np.nonzero(mask)\n    if len(xs) < 3:\n        return {"center_px": None, "orientation_deg": None, "grasp_candidates": []}\n    center = [float(xs.mean()), float(ys.mean())]\n    eigenvalues, eigenvectors = np.linalg.eigh(np.cov(np.column_stack([xs, ys]), rowvar=False))\n    orientation = None\n    if not symmetric and eigenvalues[-1] > 1.3 * max(eigenvalues[0], 1e-9):\n        axis = eigenvectors[:, -1]\n        orientation = float(np.degrees(np.arctan2(axis[1], axis[0])) % 180)\n    distance = cv2.distanceTransform(mask, cv2.DIST_L2, 5)\n    _, radius, _, point = cv2.minMaxLoc(distance)\n    candidates = []\n    if radius >= clearance_px:\n        candidates.append({"position_px": list(point), "clearance_px": float(radius),\n                           "orientation_deg": orientation, "validated": False})\n    return {"center_px": center, "orientation_deg": orientation, "grasp_candidates": candidates}\n\n\ndef classify_mask_color(frame, mask, minimum_fraction=.6):\n    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)\n    # Erode edges to reduce background contamination; classify only saturated pixels.\n    interior = cv2.erode(mask.astype(np.uint8), np.ones((3, 3), np.uint8)) > 0\n    values = hsv[interior]\n    if not len(values):\n        return None, 0.0\n    ranges = {"red": ((values[:, 0] <= 8) | (values[:, 0] >= 170)),\n              "green": (values[:, 0] >= 35) & (values[:, 0] <= 85),\n              "blue": (values[:, 0] >= 86) & (values[:, 0] <= 130)}\n    votes = {key: float(np.mean(region & (values[:, 1] >= 70) & (values[:, 2] >= 45))) for key, region in ranges.items()}\n    color = max(votes, key=votes.get)\n    confidence = votes[color]\n    return (color if confidence >= minimum_fraction else None), confidence\n\n\ndef quality_crop(frame, mask):\n    ys, xs = np.nonzero(mask)\n    if not len(xs):\n        raise ValueError("Empty instance mask")\n    patch = np.where(mask[..., None] > 0, frame, 127)[ys.min():ys.max()+1, xs.min():xs.max()+1]\n    return cv2.cvtColor(cv2.resize(patch, (224, 224)), cv2.COLOR_BGR2RGB)\n\n\nclass Segmenter:\n    def __init__(self, bundle, prefer_onnx=True, confidence=.35):\n        from ultralytics import YOLO\n        self.root = Path(bundle)\n        self.metadata = json.loads((self.root / "vision_manifest.json").read_text(encoding="utf-8"))\n        if self.metadata.get("schema_version") != "industrial-vision-1.0":\n            raise ValueError("Unsupported vision artifact schema")\n        self.classes = self.metadata["classes"]\n        filename = "segmentation.onnx" if prefer_onnx and (self.root / "segmentation.onnx").exists() else "segmentation.pt"\n        if filename.endswith(".pt") and self.metadata.get("model_family") == "yoloe":\n            from ultralytics import YOLOE\n            self.model = YOLOE(str(self.root / filename), task="segment")\n        else:\n            self.model = YOLO(str(self.root / filename), task="segment")\n        self.confidence = confidence\n        self.quality_session = None\n        if self.metadata.get("quality_enabled"):\n            import onnxruntime as ort\n            self.quality_session = ort.InferenceSession(str(self.root / "quality.onnx"), providers=["CPUExecutionProvider"])\n\n    def predict(self, frame):\n        # PT otherwise uses minimal rectangular padding, unlike static square ONNX.\n        # YOLOE also defaults to agnostic NMS while YOLO(onnx) does not.\n        prediction = self.model.predict(frame, device="cpu", imgsz=self.metadata.get("imgsz", 640),\n            conf=self.confidence, rect=False, agnostic_nms=self.metadata.get("model_family") == "yoloe",\n            iou=.7, max_det=300, retina_masks=False, augment=False, verbose=False)[0]\n        objects = []\n        if prediction.masks is None:\n            return objects\n        masks = prediction.masks.data.cpu().numpy()\n        height, width = frame.shape[:2]\n        for i, (box, cls, confidence) in enumerate(zip(prediction.boxes.xyxy.cpu().numpy(), prediction.boxes.cls.cpu().numpy(), prediction.boxes.conf.cpu().numpy())):\n            # Ultralytics masks may be in letterboxed coordinates, not frame coordinates.\n            from ultralytics.utils.ops import scale_masks\n            import torch\n            restored = scale_masks(torch.as_tensor(masks[i:i+1, None],dtype=torch.float32), (height, width))[0, 0].numpy()\n            mask = (restored > .5).astype(np.uint8)\n            if not mask.any():\n                continue\n            x1, y1, x2, y2 = box.tolist()\n            if int(cls) != cls or not 0 <= int(cls) < len(self.classes):\n                raise ValueError("Prediction class ID does not match the exported class mapping")\n            object_type = self.classes[int(cls)]\n            color, color_confidence = classify_mask_color(frame, mask)\n            geometry = mask_geometry(mask, symmetric=object_type in ("washer", "nut"))\n            if geometry["center_px"] is None:\n                continue\n            quality, quality_confidence = "unknown", 0.0\n            if self.quality_session is not None:\n                crop = quality_crop(frame, mask).astype(np.float32) / 255\n                x = ((crop - [.485, .456, .406]) / [.229, .224, .225]).transpose(2, 0, 1)[None].astype(np.float32)\n                logits = self.quality_session.run(None, {self.quality_session.get_inputs()[0].name: x})[0][0]\n                p = np.exp(logits - logits.max()); p /= p.sum()\n                quality_confidence = float(p.max())\n                if quality_confidence >= self.metadata.get("quality_threshold", .8):\n                    quality = QUALITY[int(p.argmax())]\n            partial = x1 <= 1 or y1 <= 1 or x2 >= width-1 or y2 >= height-1\n            if partial:\n                quality = "unknown"\n                geometry["grasp_candidates"] = []\n            objects.append({"type": object_type, "confidence": float(confidence), "mask": mask,\n                "bbox": [int(x1), int(y1), int(x2-x1), int(y2-y1)], "color": color,\n                "color_confidence": color_confidence, "quality": quality,\n                "quality_confidence": quality_confidence, "partial": partial, **geometry})\n        return objects\n\n\ndef draw_instances(frame, objects):\n    out = frame.copy()\n    for i, obj in enumerate(objects):\n        mask = obj["mask"] > 0\n        color = np.array([(53*i+60) % 255, (97*i+100) % 255, (131*i+180) % 255])\n        out[mask] = (.55*out[mask] + .45*color).astype(np.uint8)\n        x, y, w, h = obj["bbox"]\n        cv2.rectangle(out, (x, y), (x+w, y+h), color.tolist(), 2)\n        cv2.putText(out, f"{obj[\'type\']} {obj[\'confidence\']:.2f} {obj[\'quality\']}", (x, max(16,y-5)), cv2.FONT_HERSHEY_SIMPLEX, .45, (255,255,255), 1)\n        for point in obj["grasp_candidates"]:\n            cv2.circle(out, tuple(map(int, point["position_px"])), 4, (0,255,255), -1)\n    return out\n', 'roboqc/industrial/sam_masks.py': '"""SAM candidate export utilities. No model loading, downloads or inference."""\n\nimport json\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\n\nfrom .vision_inference import mask_geometry\n\n\ndef save_candidates(rgb, annotations, output, minimum_area=100, maximum_fraction=.85, seed=42):\n    """Preserve all raw masks; export filtered candidates without claiming object labels."""\n    rgb = np.asarray(rgb)\n    if rgb.ndim != 3 or rgb.shape[2] != 3 or rgb.dtype != np.uint8:\n        raise ValueError("Expected a uint8 RGB image")\n    if minimum_area < 1 or not 0 < maximum_fraction <= 1:\n        raise ValueError("Require minimum_area >= 1 and 0 < maximum_fraction <= 1")\n    height,width = rgb.shape[:2]\n    masks = [np.asarray(a["segmentation"],dtype=bool) for a in annotations]\n    if any(m.shape != (height,width) for m in masks):\n        raise ValueError("Every mask must match the original image shape")\n    out = Path(output)\n    out.mkdir(parents=True,exist_ok=False)\n    np.savez_compressed(out / "raw_masks.npz",masks=np.stack(masks) if masks else np.zeros((0,height,width),bool))\n    rows = []\n    for index,(annotation,mask) in enumerate(zip(annotations,masks)):\n        area = int(mask.sum())\n        reason = "below_minimum_area" if area < minimum_area else "above_maximum_fraction" if area > maximum_fraction*height*width else None\n        ys,xs = np.nonzero(mask)\n        box = [int(xs.min()),int(ys.min()),int(xs.max()-xs.min()+1),int(ys.max()-ys.min()+1)] if area else None\n        filename = f"mask_{index:04d}.png" if reason is None else None\n        if filename:\n            if not cv2.imwrite(str(out / filename),mask.astype(np.uint8)*255):\n                raise OSError(f"Could not save mask: {out / filename}")\n        rows.append({"id":index,"raw_mask_index":index,"area_px":area,"bbox_xywh":box,\n            "model_predicted_iou":float(annotation["predicted_iou"]) if annotation.get("predicted_iou") is not None else None,\n            "stability_score":float(annotation["stability_score"]) if annotation.get("stability_score") is not None else None,\n            "detector_score":float(annotation["detector_score"]) if annotation.get("detector_score") is not None else None,\n            "prompt_label":annotation.get("prompt_label"),\n            "kept":reason is None,"filter_reason":reason,"mask_file":filename,\n            "category":"unknown","quality":"unknown",**mask_geometry(mask)})\n    colors = np.random.default_rng(seed).integers(40,240,size=(len(rows),3))\n    overlay = rgb.copy()\n    # Smaller nested regions are drawn last, but all original overlapping masks remain separate.\n    for row in sorted((r for r in rows if r["kept"]),key=lambda r:r["area_px"],reverse=True):\n        mask = masks[row["id"]]\n        overlay[mask] = (.55*overlay[mask] + .45*colors[row["id"]]).astype(np.uint8)\n    for row in rows:\n        if row["kept"]:\n            x,y,w,h = row["bbox_xywh"]\n            cv2.rectangle(overlay,(x,y),(x+w-1,y+h-1),(0,170,0),1)\n            cv2.putText(overlay,str(row["id"]),(x,max(14,y)),cv2.FONT_HERSHEY_SIMPLEX,.5,(180,0,0),1)\n    if not cv2.imwrite(str(out / "overlay.png"),cv2.cvtColor(overlay,cv2.COLOR_RGB2BGR)):\n        raise OSError("Could not save overlay")\n    result = {"schema_version":"sam-candidates-1.0","image_size":[height,width],\n        "raw_masks":len(rows),"kept_candidates":sum(r["kept"] for r in rows),"candidates":rows,\n        "filters":{"minimum_area":minimum_area,"maximum_fraction":maximum_fraction},\n        "accuracy":"not_measured","warning":"Candidates may overlap or describe subparts/background, not unique physical objects"}\n    (out / "candidates.json").write_text(json.dumps(result,indent=2),encoding="utf-8")\n    return result,overlay\n', 'roboqc/industrial/pretrained_compare.py': '"""Manual Colab comparison adapters. No imports load models or start inference."""\n\nimport hashlib\nimport os\nfrom pathlib import Path\nimport tempfile\nimport warnings\n\nimport numpy as np\n\nDEFAULT_METHODS = ["yoloe_text","sam2_auto","grounding_dino_sam2","owlv2_sam2","florence2_sam2"]\nMODEL_IDS = {"grounding_dino_sam2":"IDEA-Research/grounding-dino-base",\n             "owlv2_sam2":"google/owlv2-base-patch16-ensemble",\n             "florence2_sam2":"florence-community/Florence-2-base",\n             "sam3_text":"facebook/sam3"}\n\n\ndef crop_image(rgb, normalized_roi=None):\n    """A shared optional workspace crop; no method gets a different region."""\n    height,width = rgb.shape[:2]\n    if normalized_roi is None:\n        return rgb,(0,0,width,height)\n    roi = np.asarray(normalized_roi,dtype=float)\n    if roi.shape != (4,) or not np.isfinite(roi).all() or not (0 <= roi[0] < roi[2] <= 1 and 0 <= roi[1] < roi[3] <= 1):\n        raise ValueError("ROI must be normalized [left,top,right,bottom] inside [0,1]")\n    x1,y1,x2,y2 = int(roi[0]*width),int(roi[1]*height),int(roi[2]*width),int(roi[3]*height)\n    if x2 <= x1 or y2 <= y1:\n        raise ValueError("ROI is empty at this image resolution")\n    return rgb[y1:y2,x1:x2],(x1,y1,x2,y2)\n\n\ndef restore_masks(annotations, image_shape, roi):\n    x1,y1,x2,y2 = roi\n    restored = []\n    for annotation in annotations:\n        mask = np.asarray(annotation["segmentation"],dtype=bool)\n        if mask.shape != (y2-y1,x2-x1):\n            raise ValueError("Mask dimensions do not match the processed ROI")\n        full = np.zeros(image_shape[:2],bool)\n        full[y1:y2,x1:x2] = mask\n        restored.append(dict(annotation,segmentation=full))\n    return restored\n\n\ndef select_boxes(boxes, scores, labels, size, iou_threshold=.7, maximum=50):\n    """Class-agnostic NMS on detector boxes, not a forced object count."""\n    boxes = np.asarray(boxes,dtype=float).reshape(-1,4)\n    if len(scores) != len(boxes) or len(labels) != len(boxes):\n        raise ValueError("Mismatched detector boxes, scores and labels")\n    width,height = size\n    finite = np.isfinite(boxes).all(axis=1)\n    boxes[:,[0,2]] = boxes[:,[0,2]].clip(0,width)\n    boxes[:,[1,3]] = boxes[:,[1,3]].clip(0,height)\n    valid = [i for i,b in enumerate(boxes) if finite[i] and b[2] > b[0] and b[3] > b[1]\n             and (scores[i] is None or np.isfinite(scores[i]))]\n    order = sorted(valid,key=lambda i:float(scores[i]) if scores[i] is not None else 0,reverse=True)\n    keep = []\n    while order and len(keep) < maximum:\n        i = order.pop(0); keep.append(i)\n        survivors = []\n        for j in order:\n            intersection = np.maximum(0,np.minimum(boxes[i,2:],boxes[j,2:])-np.maximum(boxes[i,:2],boxes[j,:2])).prod()\n            area_i = (boxes[i,2:]-boxes[i,:2]).prod(); area_j = (boxes[j,2:]-boxes[j,:2]).prod()\n            if intersection/max(1,area_i+area_j-intersection) <= iou_threshold:\n                survivors.append(j)\n        order = survivors\n    return boxes[keep], [scores[i] for i in keep], [labels[i] for i in keep]\n\n\ndef mask_metrics(predicted, reference, threshold=.5):\n    """One-to-one mask matching, maximizing qualifying matches before IoU."""\n    from scipy.optimize import linear_sum_assignment\n    if any(np.asarray(m).ndim != 2 for m in [*predicted,*reference]):\n        raise ValueError("Ground truth and predictions must be 2D binary masks")\n    shapes = {np.asarray(m).shape for m in [*predicted,*reference]}\n    if len(shapes) > 1: raise ValueError("Ground-truth masks must match original image dimensions")\n    overlaps = np.zeros((len(reference),len(predicted)))\n    for i,a in enumerate(reference):\n        for j,b in enumerate(predicted):\n            overlaps[i,j] = np.logical_and(a,b).sum()/max(1,np.logical_or(a,b).sum())\n    matched = []\n    if overlaps.size:\n        # A plain maximum-IoU assignment can undercount threshold-qualified matches.\n        bonus = min(overlaps.shape)+1\n        rows,columns = linear_sum_assignment(-((overlaps >= threshold)*bonus+overlaps))\n        matched = [float(overlaps[i,j]) for i,j in zip(rows,columns) if overlaps[i,j] >= threshold]\n    tp = len(matched); fp = len(predicted)-tp; fn = len(reference)-tp\n    return {"mask_iou_threshold":threshold,"tp":tp,"fp":fp,"fn":fn,\n        "precision":tp/(tp+fp) if tp+fp else None,"recall":tp/(tp+fn) if tp+fn else None,\n        "f1":2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else None,\n        "matched_mean_iou":float(np.mean(matched)) if matched else None}\n\n\ndef _sam_model(config, device):\n    import torch\n    from huggingface_hub import hf_hub_download\n    from sam2.build_sam import build_sam2, HF_MODEL_ID_TO_FILENAMES\n    name,filename = HF_MODEL_ID_TO_FILENAMES[config["sam2_id"]]\n    # Large checkpoint reads stay on Colab\'s local disk, not the Drive mount.\n    cache = Path(config.get("sam2_cache",Path(tempfile.gettempdir()) / "roboqc_sam2_cache"))\n    cache.mkdir(parents=True,exist_ok=True)\n    revision = config["revisions"].get(config["sam2_id"],"main")\n    recovered = False\n    for attempt in range(2):\n        path = Path(hf_hub_download(config["sam2_id"],filename,revision=revision,cache_dir=str(cache)))\n        try:\n            state = torch.load(str(path),map_location="cpu",weights_only=True)\n            if not isinstance(state,dict) or not isinstance(state.get("model"),dict) or not state["model"]:\n                raise ValueError("SAM checkpoint must contain a nonempty model state dictionary")\n        except Exception as exc:\n            if attempt:\n                raise RuntimeError(f"SAM checkpoint cannot be read after a fresh download: {path}. "\n                                   "Existing files were preserved; inspect the chained error.") from exc\n            # Never overwrite/delete a possibly damaged cache entry or relax weights_only.\n            cache = Path(tempfile.mkdtemp(prefix="recovery_",dir=cache))\n            warnings.warn(f"SAM checkpoint read failed ({type(exc).__name__}: {exc}); "\n                          f"retrying once in a fresh local cache: {cache}",RuntimeWarning)\n            recovered = True\n            continue\n        break\n    config["sam2_cache"] = str(cache)  # Reuse a recovered download for later methods.\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for chunk in iter(lambda:handle.read(1024*1024),b""): digest.update(chunk)\n    # Load the already validated weights once; do not reopen the checkpoint in SAM.\n    model = build_sam2(name,ckpt_path=None,device="cpu",apply_postprocessing=False)\n    model.load_state_dict(state["model"],strict=True)\n    del state\n    return model.to(device).eval(),{"model_id":config["sam2_id"],"revision":path.parent.name,\n        "checkpoint_sha256":digest.hexdigest(),"checkpoint_path":str(path),"cache_recovered":recovered}\n\n\nclass PretrainedMethod:\n    """Load only one method per instance; caller releases it before the next method."""\n    def __init__(self, name, config, device):\n        self.name,self.config,self.device = name,config,device\n        self.provenance = {"method":name,"models":[]}\n        if name == "sam2_auto" or name.endswith("_sam2"):\n            self.sam,source = _sam_model(config,device)\n            self.provenance["models"].append(source)\n            if name == "sam2_auto":\n                from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator\n                self.generator = SAM2AutomaticMaskGenerator(self.sam,points_per_side=config["points_per_side"],\n                    points_per_batch=config["points_per_batch"],crop_n_layers=config["crop_layers"],\n                    pred_iou_thresh=config["sam_iou"],stability_score_thresh=config["sam_stability"],\n                    min_mask_region_area=0,use_m2m=True,output_mode="binary_mask")\n            else:\n                from sam2.sam2_image_predictor import SAM2ImagePredictor\n                self.segmenter = SAM2ImagePredictor(self.sam)\n        if name == "yoloe_text":\n            from ultralytics import YOLOE\n            directory = Path(config["cache"]) / "yoloe"; directory.mkdir(parents=True,exist_ok=True)\n            previous = Path.cwd()\n            try:\n                os.chdir(directory)\n                self.model = YOLOE(config["yoloe_checkpoint"],task="segment")\n                self.model.set_classes(config["prompts"],self.model.get_text_pe(config["prompts"]))\n            finally:\n                os.chdir(previous)\n            checkpoint = directory / Path(config["yoloe_checkpoint"]).name\n            self.provenance["models"].append({"checkpoint":config["yoloe_checkpoint"],\n                "sha256":hashlib.sha256(checkpoint.read_bytes()).hexdigest() if checkpoint.is_file() else None})\n        elif name in MODEL_IDS:\n            from safetensors import SafetensorError\n            from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, Florence2ForConditionalGeneration, Sam3Model\n            model_id = MODEL_IDS[name]\n            kwargs = {"cache_dir":config["cache"],"revision":config["revisions"].get(model_id,"main"),"trust_remote_code":False}\n            model_class = Florence2ForConditionalGeneration if name == "florence2_sam2" else Sam3Model if name == "sam3_text" else AutoModelForZeroShotObjectDetection\n            for attempt in range(2):\n                try:\n                    self.processor = AutoProcessor.from_pretrained(model_id,**kwargs)\n                    self.model = model_class.from_pretrained(model_id,use_safetensors=True,**kwargs)\n                except SafetensorError as exc:\n                    if attempt:\n                        raise RuntimeError(f"{model_id} weights remain unreadable after a fresh local cache "\n                                           f"download: {kwargs[\'cache_dir\']}. Previous files were preserved.") from exc\n                    cache = Path(config["cache"]); cache.mkdir(parents=True,exist_ok=True)\n                    kwargs["cache_dir"] = tempfile.mkdtemp(prefix=name + "_recovery_",dir=cache)\n                    warnings.warn(f"{model_id}: {exc}; retrying once in a fresh local cache: "\n                                  f"{kwargs[\'cache_dir\']}",RuntimeWarning)\n                    continue\n                break\n            self.model = self.model.to(device).eval()\n            self.provenance["models"].append({"model_id":model_id,"revision":getattr(self.model.config,"_commit_hash",None),\n                "cache_dir":str(kwargs["cache_dir"]),"cache_recovered":bool(attempt)})\n        elif name != "sam2_auto":\n            raise ValueError(f"Unknown comparison method: {name}")\n\n    def predict(self, rgb):\n        import torch\n        from PIL import Image\n        cfg = self.config; image = Image.fromarray(rgb); height,width = rgb.shape[:2]\n        with torch.inference_mode():\n            if self.name == "sam2_auto":\n                return self.generator.generate(rgb),{"score_type":"SAM predicted IoU; not observed accuracy"}\n            if self.name == "yoloe_text":\n                from ultralytics.utils.ops import scale_masks\n                result = self.model.predict(rgb[:,:,::-1].copy(),device=self.device,imgsz=cfg["imgsz"],\n                    conf=cfg["yoloe_threshold"],rect=False,agnostic_nms=True,iou=cfg["box_nms"],\n                    max_det=cfg["max_detections"],retina_masks=False,verbose=False)[0]\n                if result.masks is None: return [],{"score_type":"YOLO detection confidence"}\n                masks = scale_masks(result.masks.data[:,None].float().cpu(),(height,width))[:,0].numpy() > .5\n                scores = result.boxes.conf.cpu().tolist(); classes = result.boxes.cls.cpu().tolist()\n                annotations = [{"segmentation":m,"detector_score":float(s),"prompt_label":cfg["prompts"][int(c)]}\n                               for m,s,c in zip(masks,scores,classes)]\n                return annotations,{"score_type":"YOLO detection confidence; no predicted mask IoU"}\n            if self.name == "sam3_text":\n                # One concept prompt; use the first configured phrase for this adapter.\n                inputs = self.processor(images=image,text=cfg["prompts"][0],return_tensors="pt").to(self.device)\n                outputs = self.model(**inputs)\n                result = self.processor.post_process_instance_segmentation(outputs,threshold=cfg["sam3_threshold"],\n                    mask_threshold=.5,target_sizes=inputs["original_sizes"].tolist())[0]\n                return [{"segmentation":m.cpu().numpy().astype(bool),"detector_score":float(s),"prompt_label":cfg["prompts"][0]}\n                        for m,s in zip(result["masks"],result["scores"])],{"score_type":"SAM 3 instance score"}\n            boxes,scores,labels = [],[],[]\n            if self.name == "grounding_dino_sam2":\n                inputs = self.processor(images=image,text=". ".join(cfg["prompts"])+".",return_tensors="pt").to(self.device)\n                outputs = self.model(**inputs)\n                result = self.processor.post_process_grounded_object_detection(outputs,inputs["input_ids"],\n                    threshold=cfg["dino_threshold"],text_threshold=cfg["dino_text_threshold"],target_sizes=[(height,width)])[0]\n                boxes,scores,labels = result["boxes"].cpu().numpy(),result["scores"].cpu().tolist(),result["text_labels"]\n            elif self.name == "owlv2_sam2":\n                inputs = self.processor(images=image,text=[cfg["prompts"]],return_tensors="pt").to(self.device)\n                outputs = self.model(**inputs)\n                result = self.processor.image_processor.post_process_object_detection(outputs,threshold=cfg["owl_threshold"],target_sizes=[(height,width)])[0]\n                boxes,scores = result["boxes"].cpu().numpy(),result["scores"].cpu().tolist()\n                labels = [cfg["prompts"][int(i)] for i in result["labels"].cpu().tolist()]\n            elif self.name == "florence2_sam2":\n                for phrase in cfg["prompts"]:\n                    task = "<OPEN_VOCABULARY_DETECTION>"\n                    # BatchFeature casts floating image tensors, keeping token IDs integer.\n                    inputs = self.processor(images=image,text=task+phrase,return_tensors="pt").to(self.device,self.model.dtype)\n                    tokens = self.model.generate(**inputs,max_new_tokens=cfg["florence_max_tokens"],num_beams=3,do_sample=False)\n                    text = self.processor.batch_decode(tokens,skip_special_tokens=False)[0]\n                    result = self.processor.post_process_generation(text,task=task,image_size=image.size)[task]\n                    found = result.get("bboxes",[])\n                    boxes.extend(found); scores.extend([None]*len(found)); labels.extend([phrase]*len(found))\n            before_nms = len(boxes)\n            boxes,scores,labels = select_boxes(boxes,scores,labels,image.size,cfg["box_nms"],cfg["max_detections"])\n            if not len(boxes): return [],{"detector_boxes_before_nms":before_nms,"prompted_boxes":0}\n            self.segmenter.set_image(rgb)\n            annotations = []\n            # Single-box calls avoid ambiguous multi-box/multimask output dimensions.\n            for box,score,label in zip(boxes,scores,labels):\n                masks,qualities,_ = self.segmenter.predict(box=box.astype(np.float32),multimask_output=False)\n                annotations.append({"segmentation":masks[0].astype(bool),"predicted_iou":float(qualities[0]),\n                                    "detector_score":score,"prompt_label":str(label)})\n            return annotations,{"detector_boxes_before_nms":before_nms,"prompted_boxes":len(boxes),\n                                "score_type":"Detector score and SAM predicted IoU are distinct; Florence detector scores unavailable"}\n'}
RUNTIME = Path('/content/roboqc_compare_runtime')
for relative,source in SOURCES.items():
    target = RUNTIME / relative; target.parent.mkdir(parents=True,exist_ok=True)
    target.write_text(source,encoding='utf-8')
sys.path.insert(0,str(RUNTIME))
from roboqc.industrial.pretrained_compare import DEFAULT_METHODS, PretrainedMethod, crop_image, restore_masks, mask_metrics
from roboqc.industrial.sam_masks import save_candidates
if not PROMPTS or any(not isinstance(p,str) or not p.strip() for p in PROMPTS): raise ValueError('Use nonempty text prompts')
if MAX_DETECTIONS < 1 or POINTS_PER_SIDE < 1 or POINTS_PER_BATCH < 1 or MAX_MASK_FRACTION <= 0 or MAX_MASK_FRACTION > 1:
    raise ValueError('Use positive limits and a mask fraction in (0,1]')
selected = list(dict.fromkeys(METHODS + (['sam3_text'] if ENABLE_SAM3 else [])))
if not selected or any(m not in DEFAULT_METHODS+['sam3_text'] for m in selected): raise ValueError('Unknown/empty METHODS')
if len(selected) < 5: print('WARNING: fewer than five methods selected; default METHODS compares five.')
paths = select_image_paths(IMAGE_PATH,RUN)
samples = []
for index,path in enumerate(paths):
    with Image.open(path) as opened: rgb = np.asarray(ImageOps.exif_transpose(opened).convert('RGB'))
    crop,roi = crop_image(rgb,ROI_NORM)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    expected = None  # No count or mask ground truth is inferred from filenames
    samples.append({'index':index,'path':str(path),'sha256':digest,'expected_count':expected,'roi':list(roi)})
    plt.figure(figsize=(8,6)); plt.imshow(rgb); plt.title(f'{index}: {path.name} | processed ROI shown in green')
    from matplotlib.patches import Rectangle
    plt.gca().add_patch(Rectangle((roi[0],roi[1]),roi[2]-roi[0],roi[3]-roi[1],fill=False,edgecolor='green')); plt.axis('off'); plt.show()
references = {}
if GT_MANIFEST_PATH:
    gt_path = Path(GT_MANIFEST_PATH)
    manifest = json.loads(gt_path.read_text())
    for sample in samples:
        name = Path(sample['path']).name
        if name not in manifest: continue  # Missing annotation is NOT an empty ground-truth scene.
        with Image.open(sample['path']) as opened: shape = np.asarray(ImageOps.exif_transpose(opened)).shape[:2]
        masks = []
        for filename in manifest[name]:
            with Image.open(gt_path.parent / filename) as opened: mask = np.asarray(opened.convert('L')) > 0
            if mask.shape != shape or not mask.any(): raise ValueError(f'Invalid ground-truth mask: {filename}')
            masks.append(mask)
        references[sample['index']] = masks
print('Selected:',selected,'| images:',len(samples),'| ground-truth scenes:',len(references))


In [ ]:
OUT = RUN / ('pretrained_comparison_' + uuid.uuid4().hex); OUT.mkdir()
cfg = {'cache':str(CACHE),'sam2_cache':SAM2_CACHE,'revisions':MODEL_REVISIONS,'sam2_id':SAM2_ID,'yoloe_checkpoint':YOLOE_CHECKPOINT,
    'prompts':PROMPTS,'imgsz':IMGSZ,'yoloe_threshold':YOLOE_THRESHOLD,'dino_threshold':DINO_THRESHOLD,
    'dino_text_threshold':DINO_TEXT_THRESHOLD,'owl_threshold':OWL_THRESHOLD,'sam3_threshold':SAM3_THRESHOLD,
    'box_nms':BOX_NMS_IOU,'max_detections':MAX_DETECTIONS,'points_per_side':POINTS_PER_SIDE,
    'points_per_batch':POINTS_PER_BATCH,'crop_layers':CROP_N_LAYERS,'sam_iou':SAM_PRED_IOU,
    'sam_stability':SAM_STABILITY,'florence_max_tokens':FLORENCE_MAX_NEW_TOKENS}
settings = {'methods':selected,'config':cfg,'sam2_commit':SAM2_COMMIT,'seed':SEED,'device':DEVICE,
    'hardware':{'python':sys.version,'torch':torch.__version__,'cpu_threads':CPU_THREADS,
                'gpu':torch.cuda.get_device_name(0) if GPU else None},
    'roi_norm':ROI_NORM,'minimum_area':MIN_MASK_AREA,'maximum_fraction':MAX_MASK_FRACTION,'warmup':WARMUP,
    'images':samples,'ground_truth_manifest':GT_MANIFEST_PATH,'weights_in_export':False}
(OUT / 'settings.json').write_text(json.dumps(settings,indent=2))
records = []
for method in selected:
    folder = OUT / method; folder.mkdir()
    runner = None
    print('Loading',method,flush=True)
    try:
        start = time.perf_counter(); runner = PretrainedMethod(method,cfg,DEVICE)
        load_seconds = time.perf_counter()-start
        (folder / 'provenance.json').write_text(json.dumps(runner.provenance,indent=2))
        if WARMUP:
            with Image.open(samples[0]['path']) as opened: rgb = np.asarray(ImageOps.exif_transpose(opened).convert('RGB'))
            crop,_ = crop_image(rgb,ROI_NORM)
            runner.predict(crop)  # Untimed, separate from reported per-image latency.
            if GPU: torch.cuda.synchronize()
        for sample in samples:
            image_out = folder / f"image_{sample['index']:02d}"
            row = {'method':method,'image_index':sample['index'],'image':Path(sample['path']).name,
                   'load_seconds_including_downloads':load_seconds,'expected_count':sample['expected_count']}
            try:
                with Image.open(sample['path']) as opened: rgb = np.asarray(ImageOps.exif_transpose(opened).convert('RGB'))
                crop,roi = crop_image(rgb,ROI_NORM)
                if GPU: torch.cuda.synchronize()
                start = time.perf_counter(); annotations,details = runner.predict(crop)
                if GPU: torch.cuda.synchronize()
                seconds = time.perf_counter()-start
                annotations = restore_masks(annotations,rgb.shape,roi)
                fraction = MAX_MASK_FRACTION * crop.shape[0]*crop.shape[1]/(rgb.shape[0]*rgb.shape[1])
                result,_ = save_candidates(rgb,annotations,image_out,MIN_MASK_AREA,fraction,SEED)
                Image.fromarray(rgb).save(image_out / 'input.png')
                kept = [a['segmentation'] for a,r in zip(annotations,result['candidates']) if r['kept']]
                row.update(status='ok',elapsed_s=seconds,raw_masks=result['raw_masks'],kept_masks=len(kept),
                    overlay=str((image_out / 'overlay.png').relative_to(OUT)),details=details,
                    count_abs_error=abs(len(kept)-sample['expected_count']) if sample['expected_count'] is not None else None)
                row['accuracy_status'] = 'measured_on_supplied_masks' if sample['index'] in references else 'not_measured'
                if sample['index'] in references: row['metrics'] = mask_metrics(kept,references[sample['index']])
            except Exception as exc:
                row.update(status='error',error=f'{type(exc).__name__}: {exc}')
                (folder / f"error_image_{sample['index']:02d}.txt").write_text(traceback.format_exc())
                print('ERROR:',method,row['image'],row['error'],flush=True)
                traceback.print_exc(file=sys.stdout)
            records.append(row)
            (OUT / 'comparison.json').write_text(json.dumps(records,indent=2))
            pd.json_normalize(records).to_csv(OUT / 'comparison.csv',index=False)
            print(method,row['image'],row['status'],'| masks:',row.get('kept_masks'),'| seconds:',row.get('elapsed_s'),flush=True)
    except Exception as exc:
        error = f'{type(exc).__name__}: {exc}'
        (folder / 'load_or_warmup_error.txt').write_text(traceback.format_exc())
        records.append({'method':method,'status':'load_or_warmup_error','error':error})
        print('ERROR:',method,error,flush=True)
        traceback.print_exc(file=sys.stdout)
        (OUT / 'comparison.json').write_text(json.dumps(records,indent=2))
        pd.json_normalize(records).to_csv(OUT / 'comparison.csv',index=False)
    finally:
        del runner
        gc.collect()
        if GPU: torch.cuda.empty_cache()
completed = [m for m in selected if sum(r['status'] == 'ok' and r.get('method') == m for r in records) == len(samples)]
(OUT / 'run_status.json').write_text(json.dumps({'configured':selected,'completed_all_images':completed,
    'five_completed':len(completed)>=5,'accuracy_winner':None,'note':'No automatic winner; count agreement alone is not correct segmentation'},indent=2))
print('Completed all images:',completed)
if len(completed) < 5: print('WARNING: fewer than five methods completed. Inspect error files; failures were NOT replaced by zero predictions.')


In [ ]:
display(pd.json_normalize(records))
for sample in samples:
    columns = 3; count = len(selected)+1
    fig,axes = plt.subplots((count+columns-1)//columns,columns,figsize=(17,6*((count+columns-1)//columns)),squeeze=False)
    for axis in axes.flat: axis.axis('off')
    with Image.open(sample['path']) as original: axes.flat[0].imshow(ImageOps.exif_transpose(original).convert('RGB'))
    axes.flat[0].set_title(Path(sample['path']).name + ' | original')
    for axis,method in zip(list(axes.flat)[1:],selected):
        row = next((r for r in records if r['method'] == method and r.get('image_index') == sample['index']),None)
        if row and row['status'] == 'ok':
            with Image.open(OUT / row['overlay']) as view: axis.imshow(view)
            axis.set_title(f"{method}: {row['kept_masks']} masks | {row['elapsed_s']:.2f}s")
        else:
            axis.text(.05,.5,'Failed/unavailable: inspect error report',wrap=True); axis.set_title(method)
    plt.tight_layout(); plt.savefig(OUT / f"comparison_image_{sample['index']:02d}.png",dpi=120); plt.show()
print('Predicted IoU / detector confidence are not shared accuracy scales.')
print('Count error is only a diagnostic. Mask P/R/F1@IoU0.5 requires reviewed per-instance ground truth; no mAP claimed.')


## Optional ground-truth evaluation and interpreting results
To measure mask precision/recall/F1 at IoU 0.5, provide a JSON file mapping
image filenames to lists of binary PNG mask paths relative to that JSON.
Example: `{"image_test_1.jpg":["masks/part_1.png","masks/part_2.png"]}`.
Label EVERY visible target instance, not just two. Each mask must match the
full original image dimensions; nonzero pixels mean visible target surface.
Missing filename = unlabelled scene; explicit [] = a verified empty scene.
Holes stay background. Occluded surfaces are not invented. Supply masks before
running the comparison; the notebook never creates ground truth from predictions.

Inspect whole-part coverage, merged/fragmented instances and background masks.
Lowering thresholds adds candidates, not necessarily correct pieces. If using
a workspace crop, ensure it includes all annotated target pieces for fair recall.
Latency excludes loading/downloads, warmup and export, but includes both detector
and SAM for two-stage methods. Three images do NOT establish production performance.
If GPU memory is insufficient: try SAM2_ID='facebook/sam2.1-hiera-small' for ALL
methods using SAM; lower POINTS_PER_BATCH; rerun as a new comparison.
SAM 3 requires approval at https://huggingface.co/facebook/sam3 and a token in
Colab Secrets named HF_TOKEN. No secrets are saved to exported reports.


In [ ]:
shutil.copy2(RUN / 'environment.txt',OUT / 'requirements-colab.txt')
shutil.copytree(RUNTIME / 'roboqc',OUT / 'runtime/roboqc')
(OUT / 'README.txt').write_text('Pretrained development comparison, not an industrial detector bundle. No training. All weights use local MODEL_CACHE/SAM2_CACHE and disappear on runtime reset; results stay in the configured run folder. Cache recovery preserves previous files. Model revisions/checksums/paths recorded where available. No accuracy claims without supplied masks. Verify user image and model redistribution rights.')
archive = shutil.make_archive(str(OUT),'zip',OUT)
print('Comparison ZIP (reports, images, masks and runtime; no weights):',archive)
